In [1]:
import pandas as pd

df = pd.read_csv("../data/retail_raw.csv", parse_dates=["InvoiceDate"])
df = df.rename(columns={"Customer ID": "CustomerID"})

print("shape:", df.shape)
print("duplicates:", df.duplicated().sum())
print("cancellations:", df["Invoice"].astype(str).str.startswith("C").sum())
print("zero/negative quantity:", (df["Quantity"] <= 0).sum())
print("zero/negative price:", (df["Price"] <= 0).sum())

shape: (1067371, 8)
duplicates: 34335
cancellations: 19494
zero/negative quantity: 22950
zero/negative price: 6207


In [2]:
clean = df.drop_duplicates().copy()
clean["Invoice"] = clean["Invoice"].astype(str)

clean = clean[~clean["Invoice"].str.startswith("C")]
clean = clean[(clean["Quantity"] > 0) & (clean["Price"] > 0)]

non_product = ["POST", "D", "M", "BANK CHARGES", "PADS", "DOT", "CRUK", "C2"]
clean = clean[~clean["StockCode"].astype(str).str.upper().isin(non_product)]

clean["Revenue"] = clean["Quantity"] * clean["Price"]
clean["CustomerID"] = clean["CustomerID"].astype("Int64")

print("rows after cleaning:", clean.shape[0])

rows after cleaning: 1003473


In [ ]:
clean.to_csv("../data/retail_clean.csv", index=False)

print("missing CustomerID share:", round(clean["CustomerID"].isna().mean(), 3))
print("date range:", clean["InvoiceDate"].min(), "->", clean["InvoiceDate"].max())
print("total revenue:", round(clean["Revenue"].sum(), 2))

missing CustomerID share: 0.226
date range: 2009-12-01 07:45:00 -> 2011-12-09 12:50:00
total revenue: 19687139.48


: 